## Multi-Head attention With weight split
in this file we implement it step by step .
wq1,wq2 -> for 2 head attention and matrix multiplication is high bec , we have more matrix as the Head increases

Head inc -> weight matrix inc -> matrix mul inc .

we combine all in one matrix wq1 , wq2 = wq
one handle all the matrix operations

## Step-1 starts with input

In [545]:
import torch 
inputs=torch.tensor([[0.1,0.2,0.3,0.4,0.5,0.6], # the
                    [0.1,0.2,0.3,0.4,0.5,0.6],  # cat
                    [0.1,0.2,0.3,0.4,0.5,0.6]]) # sleeps
print(inputs.shape) # shape 
d_in=inputs.shape[-1]
num_token=inputs.shape[0]
print("d_in=",d_in," , ","num_token=",num_token)
inputs=torch.stack((inputs,inputs),dim=0)
b=inputs.shape[0]
print("b=",b)

torch.Size([3, 6])
d_in= 6  ,  num_token= 3
b= 2


## Step-2  Decide d_out,num_heads

In [546]:
d_out=6 ## dim of output context vector
num_heads=2 ## we use 2 attention head
print("d_out=",d_out,"num_heads=",num_heads)


d_out= 6 num_heads= 2


## Step-3 Initalize trainable weights


In [547]:
import torch.nn as nn
torch.manual_seed(123)
wk=nn.Parameter(torch.randn(d_in,d_out)) ##(6,6)
wq=nn.Parameter(torch.randn(d_in,d_out))
wv=nn.Parameter(torch.randn(d_in,d_out))
wk.shape


torch.Size([6, 6])

## Step-4 calculate key,value,query

In [548]:
key=inputs@wk
value=inputs@wv
query=inputs@wq
key.shape

torch.Size([2, 3, 6])

In [549]:
key

tensor([[[ 0.6170,  0.6156, -0.3318, -2.0460, -0.5923, -2.0246],
         [ 0.6170,  0.6156, -0.3318, -2.0460, -0.5923, -2.0246],
         [ 0.6170,  0.6156, -0.3318, -2.0460, -0.5923, -2.0246]],

        [[ 0.6170,  0.6156, -0.3318, -2.0460, -0.5923, -2.0246],
         [ 0.6170,  0.6156, -0.3318, -2.0460, -0.5923, -2.0246],
         [ 0.6170,  0.6156, -0.3318, -2.0460, -0.5923, -2.0246]]],
       grad_fn=<UnsafeViewBackward0>)

## Step-5 Unroll last dim of key,value,query to  include num_head,head_dim

head_dim=d_out/num_head


In [550]:
key.shape # 2= b
# 3=num_token 
# 6=d_out

torch.Size([2, 3, 6])

In [551]:
head_dim=d_out/num_heads
int(head_dim)

3

d_out=num_heads * head_dim

(b,num_token,d_out)-->
(b,num_token,num_head,head_dim)

In [552]:
print(b,num_token,num_heads,head_dim)

2 3 2 3.0


In [553]:
keys=key.view((b,num_token,num_heads,int(head_dim))) # view take only int
values=value.view((b,num_token,num_heads,int(head_dim))) # view take only int
querys=query.view((b,num_token,num_heads,int(head_dim))) # view take only int
keys

tensor([[[[ 0.6170,  0.6156, -0.3318],
          [-2.0460, -0.5923, -2.0246]],

         [[ 0.6170,  0.6156, -0.3318],
          [-2.0460, -0.5923, -2.0246]],

         [[ 0.6170,  0.6156, -0.3318],
          [-2.0460, -0.5923, -2.0246]]],


        [[[ 0.6170,  0.6156, -0.3318],
          [-2.0460, -0.5923, -2.0246]],

         [[ 0.6170,  0.6156, -0.3318],
          [-2.0460, -0.5923, -2.0246]],

         [[ 0.6170,  0.6156, -0.3318],
          [-2.0460, -0.5923, -2.0246]]]], grad_fn=<ViewBackward0>)

In [554]:
values.shape # b,num_tok,num_head,head_dim

torch.Size([2, 3, 2, 3])

## Step-6 Group by the head 
now the group by the token 
so to make group by the head we have to transpose 

In [555]:
keys=keys.transpose(1,2)
values=values.transpose(1,2)
querys=querys.transpose(1,2)
keys

tensor([[[[ 0.6170,  0.6156, -0.3318],
          [ 0.6170,  0.6156, -0.3318],
          [ 0.6170,  0.6156, -0.3318]],

         [[-2.0460, -0.5923, -2.0246],
          [-2.0460, -0.5923, -2.0246],
          [-2.0460, -0.5923, -2.0246]]],


        [[[ 0.6170,  0.6156, -0.3318],
          [ 0.6170,  0.6156, -0.3318],
          [ 0.6170,  0.6156, -0.3318]],

         [[-2.0460, -0.5923, -2.0246],
          [-2.0460, -0.5923, -2.0246],
          [-2.0460, -0.5923, -2.0246]]]], grad_fn=<TransposeBackward0>)

## Step-7 Find attention Score
(b,head,token,dim-head) (b,head,dim-head,token)
do not focus on the b and the head 
treat it as single head
if single head then the dim is like this(num_token,head_dim)=querys
(num_token,head_dim)=keys
(head_dim,num_token)=keys.T

In [556]:
attention_score=querys@keys.transpose(2,3) 
## (2,2,3,3) (2,2,3,3)

attention_score

tensor([[[[ 0.7866,  0.7866,  0.7866],
          [ 0.7866,  0.7866,  0.7866],
          [ 0.7866,  0.7866,  0.7866]],

         [[-1.7886, -1.7886, -1.7886],
          [-1.7886, -1.7886, -1.7886],
          [-1.7886, -1.7886, -1.7886]]],


        [[[ 0.7866,  0.7866,  0.7866],
          [ 0.7866,  0.7866,  0.7866],
          [ 0.7866,  0.7866,  0.7866]],

         [[-1.7886, -1.7886, -1.7886],
          [-1.7886, -1.7886, -1.7886],
          [-1.7886, -1.7886, -1.7886]]]], grad_fn=<UnsafeViewBackward0>)

## Step-8 Attention Weights

In [557]:
keys.shape[-1] ## head dim

3

In [558]:
context_len=inputs[1].shape[0]
context_len

3

#### a. mask

In [559]:
mask=torch.triu(torch.ones(context_len,context_len),diagonal=1)
mask

tensor([[0., 1., 1.],
        [0., 0., 1.],
        [0., 0., 0.]])

In [560]:
mask.bool()

tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])

In [561]:
attention_score_mask=attention_score.masked_fill(mask.bool(),-torch.inf)
attention_score_mask

tensor([[[[ 0.7866,    -inf,    -inf],
          [ 0.7866,  0.7866,    -inf],
          [ 0.7866,  0.7866,  0.7866]],

         [[-1.7886,    -inf,    -inf],
          [-1.7886, -1.7886,    -inf],
          [-1.7886, -1.7886, -1.7886]]],


        [[[ 0.7866,    -inf,    -inf],
          [ 0.7866,  0.7866,    -inf],
          [ 0.7866,  0.7866,  0.7866]],

         [[-1.7886,    -inf,    -inf],
          [-1.7886, -1.7886,    -inf],
          [-1.7886, -1.7886, -1.7886]]]], grad_fn=<MaskedFillBackward0>)

#### b. softmax

In [562]:
keys.shape[-1]

3

In [563]:
head_dim

3.0

In [564]:
attention_wt=torch.softmax(attention_score_mask/keys.shape[-1]**0.5,dim=-1)
attention_wt

tensor([[[[1.0000, 0.0000, 0.0000],
          [0.5000, 0.5000, 0.0000],
          [0.3333, 0.3333, 0.3333]],

         [[1.0000, 0.0000, 0.0000],
          [0.5000, 0.5000, 0.0000],
          [0.3333, 0.3333, 0.3333]]],


        [[[1.0000, 0.0000, 0.0000],
          [0.5000, 0.5000, 0.0000],
          [0.3333, 0.3333, 0.3333]],

         [[1.0000, 0.0000, 0.0000],
          [0.5000, 0.5000, 0.0000],
          [0.3333, 0.3333, 0.3333]]]], grad_fn=<SoftmaxBackward0>)

In [565]:
attention_score_mask[0][0]

tensor([[0.7866,   -inf,   -inf],
        [0.7866, 0.7866,   -inf],
        [0.7866, 0.7866, 0.7866]], grad_fn=<SelectBackward0>)

In [566]:
torch.softmax(attention_score_mask[0][0],dim=-1)

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]], grad_fn=<SoftmaxBackward0>)

In [567]:
attention_score_mask[0][1]

tensor([[-1.7886,    -inf,    -inf],
        [-1.7886, -1.7886,    -inf],
        [-1.7886, -1.7886, -1.7886]], grad_fn=<SelectBackward0>)

In [568]:
torch.softmax(attention_score_mask[0][1],dim=-1)

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]], grad_fn=<SoftmaxBackward0>)

bec of same value of each attention socre thats why the head 1 and 2 both have same attention wt

## Step-9 Context Vector
attention_wt=(b,num_head,num_token,num_token)
values=(b,num_head,num_token,head_dim)
igonre the b and num head only focus on the 
(2,3) dim of the each matrix

In [569]:
context_vec=attention_wt@values
context_vec

tensor([[[[-0.2089,  1.4501, -0.7853],
          [-0.2089,  1.4501, -0.7853],
          [-0.2089,  1.4501, -0.7853]],

         [[ 0.0446,  0.3964, -0.6694],
          [ 0.0446,  0.3964, -0.6694],
          [ 0.0446,  0.3964, -0.6694]]],


        [[[-0.2089,  1.4501, -0.7853],
          [-0.2089,  1.4501, -0.7853],
          [-0.2089,  1.4501, -0.7853]],

         [[ 0.0446,  0.3964, -0.6694],
          [ 0.0446,  0.3964, -0.6694],
          [ 0.0446,  0.3964, -0.6694]]]], grad_fn=<UnsafeViewBackward0>)

## Step-10 Reformate context vec
bec now context vec is group by the head 
we want to group by the token

[token1_conect_vec]->head1

[token1_conect_vec]->head2

we want like this

conext_vec=(b,num_head,num_token,head_dim)
convert into
(b,num_token,num_head,head_dim)

In [570]:
context_vec_transform=context_vec.transpose(1,2)
context_vec_transform

tensor([[[[-0.2089,  1.4501, -0.7853],
          [ 0.0446,  0.3964, -0.6694]],

         [[-0.2089,  1.4501, -0.7853],
          [ 0.0446,  0.3964, -0.6694]],

         [[-0.2089,  1.4501, -0.7853],
          [ 0.0446,  0.3964, -0.6694]]],


        [[[-0.2089,  1.4501, -0.7853],
          [ 0.0446,  0.3964, -0.6694]],

         [[-0.2089,  1.4501, -0.7853],
          [ 0.0446,  0.3964, -0.6694]],

         [[-0.2089,  1.4501, -0.7853],
          [ 0.0446,  0.3964, -0.6694]]]], grad_fn=<TransposeBackward0>)

## Step-11 Combine Head
we want to combine the head now the look like this 

[token1_conect_vec]->head1 ()

[token1_conect_vec]->head2

we want into this 

[token1_conect_vec_head1 , token1_conect_vec_head2]

(6-dim vector become)
d_out=6 .  we define it in the starting

we get the combined comtext vector of both head


In [571]:
context_vec_transform.contiguous().view(b,num_token,d_out)

tensor([[[-0.2089,  1.4501, -0.7853,  0.0446,  0.3964, -0.6694],
         [-0.2089,  1.4501, -0.7853,  0.0446,  0.3964, -0.6694],
         [-0.2089,  1.4501, -0.7853,  0.0446,  0.3964, -0.6694]],

        [[-0.2089,  1.4501, -0.7853,  0.0446,  0.3964, -0.6694],
         [-0.2089,  1.4501, -0.7853,  0.0446,  0.3964, -0.6694],
         [-0.2089,  1.4501, -0.7853,  0.0446,  0.3964, -0.6694]]],
       grad_fn=<ViewBackward0>)

In [572]:
context_vec_transform[0][0] # token 1 context vec by head 1 and 2

tensor([[-0.2089,  1.4501, -0.7853],
        [ 0.0446,  0.3964, -0.6694]], grad_fn=<SelectBackward0>)